# interp-engine on Colab — vLLM static backend

**Arrived from the diagram's Notebook button?** The snippet you clicked is on your clipboard.
Run the install cell, then paste it into the last cell and run that.

[`interp-engine`](https://github.com/decoderesearch/interp-engine) reads activations out of a
transformer at 34 standardized points — `resid_post.10`, `mlp_act.3`, `z.7` — and an address means
the same tensor on every architecture it supports. This notebook is the runnable half of the point
diagram at **[interp-engine.org](https://interp-engine.org)**, which is the cheat sheet for where
those points sit and what other stacks call them.

Every card on the diagram has its own URL, so the one this snippet came from can be reopened or
sent to someone: `https://interp-engine.org/?arch=Qwen3ForCausalLM&point=resid_post.2` is
`resid_post` at layer 2 on Qwen 3, and `?vs=` beside it draws a second architecture to compare
against. The same table as markdown, with which backend serves what, is in
[SUPPORTED_POINTS.md](https://github.com/decoderesearch/interp-engine/blob/main/docs/SUPPORTED_POINTS.md).

## What makes this backend different

`backend="vllm-static"` declares its taps **at load** and bakes them into CUDA graphs, so it replays
the graphs instead of running the Python forward — between 4x and 11x the decode throughput of the
hooked backend, without giving up capture or steering.

The trade is in that word *declares*. The tap set is fixed when the graphs are recorded, so this
engine serves exactly the points named in `static_points=` and refuses everything else. **This is why
the snippet you pasted names its point twice** — once in `static_points=` at load, and once in the
capture call:

```python
point = Address("resid_post", 10)
model = load_model("Qwen/Qwen3-8B", backend="vllm-static", static_points=[point])
await model.warmup()
acts = await model.capture(model.to_tokens("Hello, world")[0].tolist(), [point])
```

That `await` is the second thing about the paste worth knowing. Colab runs every cell inside an event
loop, where the sync `run_with_cache` raises `NestedEventLoop` rather than nest a second one, so the
Notebook button copies the awaited form of the card's `vllm static` tab — the same points through the
same code path, with a comment at the top saying so. Colab's kernel runs top-level `await` directly,
so there is no `asyncio.run` to add.

Asking a running model for a point it did not declare is an error, not a short result — and the
error carries the `load_model` call that would have served it. There is no way to add a tap without
reloading, which is the honest shape of the mechanism rather than a limitation of the API.

`static_points="auto"` is the useful default when you do not have a list yet: `resid_post` at every
layer, to read *and* to write. Omitting `static_points` altogether gets you exactly that.

Two points this backend cannot serve at any tap set: `embeddings` and `final_norm` hang off the
trunk rather than a decoder layer, and `attn_scores` / `attn_probs` are rebuilt off-kernel — declare
`Address("attn", layer)` and let `capture_attention` recompute the matrix. For anything else, or for
a point set you only discover per request, switch the card to its `vllm` tab and open
[the hooked notebook](https://colab.research.google.com/github/decoderesearch/interp-engine/blob/main/notebooks/interp_engine_vllm.ipynb),
which serves every point and chooses them per call.

## Pick a GPU runtime first

**Runtime → Change runtime type → T4 GPU**, then Save. vLLM is CUDA-only: on a CPU runtime the
install resolves and then `load_model` has no worker to start. This notebook asks for a GPU in its
metadata, so a fresh copy usually has one already — worth confirming, since a copy that has been
saved and reopened keeps whatever runtime it was last connected to.

A free T4 has 15 GB and no bf16, which fits checkpoints up to roughly 7B in fp16 — and this backend
wants **more** of that card than the hooked one, because the static buffers have to fit alongside
the graphs. The engine steps `max_num_batched_tokens` down to make room and refuses rather than
OOM-ing during graph capture, so an out-of-memory here is a refusal with a number in it. Three knobs
are worth knowing before the first one: `gpu_memory_utilization=0.8` leaves vLLM less than the 90%
it claims by default, `max_model_len=2048` shrinks the KV cache, and naming a short
`static_points=[...]` list rather than `"auto"` is the cheapest saving of the three — one layer's tap
instead of every layer's. `static_writes=[]` buys back more, at the cost of steering.

For a backend that needs no GPU at all, switch the card to its `eager` tab and open
[the eager notebook](https://colab.research.google.com/github/decoderesearch/interp-engine/blob/main/notebooks/interp_engine_eager.ipynb)
instead.

In [ ]:
# Several minutes. The vLLM wheel and its CUDA dependencies are a few GB, and pip replaces
# the torch Colab ships with the one vLLM pins -- which is why it may end by telling you to
# restart the session. Do that (Runtime -> Restart session), then carry on from the next
# cell: the packages are installed, and only this kernel needed replacing.
#
# The uninstall is what keeps torch and its companions one set. Colab builds all four for
# its own CUDA, and pip leaves alone any whose version already satisfies vLLM's pin -- so
# torch arrives from PyPI built for CUDA 13 while torchaudio stays on Colab's build for
# 12.8, and torchaudio then raises on import from inside warmup(). Removing them first
# means the set that comes back was resolved together.
!pip uninstall -q -y torch torchvision torchaudio torchcodec
!pip install -q "interp-engine[vllm]"

In [ ]:
# Gated checkpoints -- Gemma and Llama, among others -- need a Hugging Face token on an
# account that has accepted the model's terms. Keep it in Colab's Secrets (the key icon in
# the left sidebar) as HF_TOKEN with "Notebook access" on, and uncomment these two lines.
#
# Not as a literal in the cell: a notebook is the thing you share, and a pasted token is
# what leaks with it.

# from google.colab import userdata
# from huggingface_hub import login; login(userdata.get("HF_TOKEN"))

In [ ]:
# Paste the snippet here (Ctrl+V, or Cmd+V on a Mac), then run this cell.
#
# The diagram's Notebook button put it on your clipboard on the way in. If the clipboard
# turns out to be empty -- some browsers refuse the write outright -- reopen the point at
# interp-engine.org and press Copy beside the tabs. Copy gives the sync form, which a cell
# running inside an event loop refuses: await the methods instead, as the notes above do.
#
# The first run downloads the weights and then records the CUDA graphs over the taps the
# snippet declared, so it is minutes slower than every run after it in the same session.
# The graph capture is the part the hooked backend does not pay for, and the part every
# later request is faster for.
#
# To read a second point, edit static_points= and re-run this cell. A tap cannot be added
# to graphs that are already recorded, so it is a reload -- which is what the error you
# would otherwise get says, with the call to make.